
## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.

Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

```python
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
```
````


In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

response = model.invoke("Why do parrots talk?")

print(response.content)

c:\Users\DELL\Documents\AgenticAI\langchain_updated\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


<think>
Okay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They're birds known for their ability to mimic human speech. But why do they do that? Is it instinctual, or do they learn it?

First, maybe it's about communication. Parrots are social animals, right? They live in flocks in the wild, so vocalizations are probably important for them. But when they're with humans, they might use speech to interact. So maybe they talk to communicate with their human caregivers, similar to how they would with other parrots.

I remember reading that parrots have a part of their brain called the "parrot brain" that's involved in vocal learning. It's a bit like how humans have a specific area for language. So maybe their brain structure allows them to learn sounds, including human speech.

Another angle is that talking helps them bond with their owners. If a parrot can mimic words, the owner might be more engaged, leading to positive reinforcement like

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "What is the weather in Chennai?"
)

print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Chennai. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Chennai, I need to call this function with "Chennai" as the location. I\'ll make sure the arguments are correctly formatted in JSON within the tool_call tags.\n', 'tool_calls': [{'id': 'x5ewzwxdy', 'function': {'arguments': '{"location":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 153, 'total_tokens': 248, 'completion_time': 0.146200722, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006270448, 'prompt_tokens_details': None, 'queue_time': 0.053901082, 'total_time': 0.15247117}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider'

In [3]:
response = model_with_tools.invoke(
    "What's the weather like in Boston?"
)

print(response)

# View tool calls made by the model
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston.\n", 'tool_calls': [{'id': 'ffr83s1mn', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 154, 'total_tokens': 227, 'completion_time': 0.112000212, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.007354505, 'prompt_tokens_details': None, 'queue_time': 0.285713484, 'total_time': 0.119354717}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e5415-750c-7820-ac31-68410fe7e370-0' tool_calls=[{'name': 'get_weather', 'args': {'locat

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [
    {"role": "user", "content": "What's the weather in Boston?"}
]

ai_msg = model_with_tools.invoke(messages)

messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)

    # Add tool result to conversation
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

print(final_response.content)

# Example:
# "The weather in Boston is sunny."

The weather in Boston is sunny ☀️.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function\'s parameters. The required parameter is location, which should be a string. Boston is the location here, so I\'ll call the function with "Boston" as the argument. Make sure the JSON is correctly formatted with the function name and arguments. No other tools are available, so this is the only function to use.\n', 'tool_calls': [{'id': '4bgpsx8bz', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 153, 'total_tokens': 267, 'completion_time': 0.201763451, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.006747431, 'prompt_tokens_details': None, 'queue_time': 0.054912589, 'total_time': 0.208510882}, 'm